In [ ]:
#Install DocTR and dependencies
!pip install "python-doctr[torch]" pdf2image pillow --quiet
!sudo apt-get install -y poppler-utils


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.4/288.4 kB 20.1 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 186 kB of 

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload() 
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf" 
    
    print(f"Running locally, using: {pdf_path}")

In [ ]:
#Convert PDF → page images (PNG)
import os
from pdf2image import convert_from_path

PAGES_DIR = "pages"
os.makedirs(PAGES_DIR, exist_ok=True)

images = convert_from_path(pdf_path, dpi=200)
image_paths = []

for i, img in enumerate(images, start=1):
    path = os.path.join(PAGES_DIR, f"page_{i:03}.png")
    img.save(path)
    image_paths.append(path)

print("Total pages:", len(image_paths))
image_paths[:5]



In [ ]:
#Load DocTR OCR model
from doctr.models import ocr_predictor

# This loads a detection + recognition OCR pipeline (DBNet + CRNN/SAR)
model = ocr_predictor(pretrained=True)
print(type(model))


In [ ]:
#Run OCR on all pages
from doctr.io import DocumentFile

# Load images as a DocTR Document
doc = DocumentFile.from_images(image_paths)

# Run OCR
result = model(doc)
type(result)


In [ ]:
#Convert OCR result → Markdown text
def doctr_to_markdown(result):
    lines = []
    for page_idx, page in enumerate(result.pages, start=1):
        lines.append(f"# Page {page_idx}")
        for block in page.blocks:
            for line in block.lines:
                text = " ".join(word.value for word in line.words)
                if text.strip():
                    lines.append(text)
        lines.append("")  # blank line between pages
    return "\n".join(lines)

markdown_text = doctr_to_markdown(result)

print(markdown_text[:10000])


In [ ]:
#Save Markdown in Colab
import os

base_name = os.path.splitext(os.path.basename(pdf_path))[0]
md_filename = base_name + ".md"
md_path = os.path.join("/content", md_filename)

with open(md_path, "w", encoding="utf-8") as f:
    f.write(markdown_text)

print("Markdown saved at:", md_path)


In [ ]:
#Download Markdown
from google.colab import files

files.download(md_path)
